# Day 2: API + AI Integration

## Goal

Extend the Day 1 API so the `/query` endpoint calls an LLM and returns an AI-generated response.

By the end of this notebook you will have:

- a FastAPI app
- one `/query` endpoint
- support for **OpenAI** or **Ollama**
- a notebook test that sends a request through the API and gets an LLM response back

This notebook assumes the Day 1 API concepts are already clear.

## Step 0: Sync Once and Configure Environment Variables

The required packages are already included in `pyproject.toml`, so learners only need:

```bash
uv sync
```

Then set the values you want in `.env`.

Example:

```text
OPENAI_API_KEY=your_openai_key
OPENAI_MODEL=your_openai_model
OLLAMA_MODEL=llama3.2
OLLAMA_BASE_URL=http://localhost:11434/v1
```

If you plan to use Ollama, make sure Ollama is running locally and that you have already pulled a model.

In [ ]:
import os
from typing import Optional

from dotenv import load_dotenv
from fastapi import FastAPI, HTTPException, Body
from fastapi.testclient import TestClient
from openai import OpenAI

print("Imports loaded successfully.")


In [ ]:
load_dotenv(override=True)

config_summary = {
    "OPENAI_API_KEY_present": bool(os.getenv("OPENAI_API_KEY")),
    "OPENAI_MODEL": os.getenv("OPENAI_MODEL"),
    "OLLAMA_MODEL": os.getenv("OLLAMA_MODEL"),
    "OLLAMA_BASE_URL": os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
}

config_summary

## Step 1: Define the API Contract

This time the request contains the `provider`, the `prompt`, and optionally a `system_prompt` and `model`.

Supported providers:

- `openai`
- `ollama`

In [ ]:
def parse_query_payload(payload: dict) -> dict:
    # Basic validation without Pydantic for beginners.
    provider = payload.get("provider")
    if provider not in {"openai", "ollama"}:
        raise ValueError("provider must be 'openai' or 'ollama'")

    prompt = payload.get("prompt")
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be a non-empty string")

    system_prompt = payload.get("system_prompt")
    if system_prompt is None:
        system_prompt = "You are a helpful assistant."
    elif not isinstance(system_prompt, str):
        raise ValueError("system_prompt must be a string")

    model = payload.get("model")
    if model is not None and not isinstance(model, str):
        raise ValueError("model must be a string or null")

    return {
        "provider": provider,
        "prompt": prompt,
        "system_prompt": system_prompt,
        "model": model,
    }


parse_query_payload({"provider": "openai", "prompt": "hi"})


## Step 2: Create Helper Functions for OpenAI and Ollama

We will use the OpenAI Python client for both providers.

- For `openai`, we use the normal API key.
- For `ollama`, we point the client at the local OpenAI-compatible base URL.

In [ ]:
def resolve_model(provider: str, model_override: Optional[str] = None) -> str:
    if model_override:
        return model_override

    if provider == "openai":
        model = os.getenv("OPENAI_MODEL")
        if not model:
            raise ValueError("OPENAI_MODEL is not set in .env")
        return model

    if provider == "ollama":
        model = os.getenv("OLLAMA_MODEL")
        if not model:
            raise ValueError("OLLAMA_MODEL is not set in .env")
        return model

    raise ValueError("Unsupported provider. Use 'openai' or 'ollama'.")


def create_llm_client(provider: str) -> OpenAI:
    if provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is not set in .env")
        return OpenAI(api_key=api_key)

    if provider == "ollama":
        base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
        return OpenAI(base_url=base_url, api_key="ollama")

    raise ValueError("Unsupported provider. Use 'openai' or 'ollama'.")


print("Helpers ready.")


In [ ]:
def ask_llm(
    provider: str,
    prompt: str,
    system_prompt: Optional[str] = None,
    model: Optional[str] = None,
) -> dict:
    client = create_llm_client(provider)
    selected_model = resolve_model(provider, model)

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=selected_model,
        messages=messages,
    )

    answer = response.choices[0].message.content or ""

    return {
        "provider": provider,
        "model": selected_model,
        "answer": answer,
    }


print("LLM call function ready.")


## Step 3: Put the LLM Call Behind the API Endpoint

The endpoint will receive the request, call the selected provider, and return the AI response as JSON.

In [ ]:
app = FastAPI(title="Day 2 Query API")


@app.post("/query")
def query_llm(payload: dict = Body(...)):
    try:
        parsed = parse_query_payload(payload)
        result = ask_llm(
            provider=parsed["provider"],
            prompt=parsed["prompt"],
            system_prompt=parsed["system_prompt"],
            model=parsed["model"],
        )
        return {
            "message": "LLM response generated",
            "data": result,
        }
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"LLM call failed: {exc}") from exc


print("FastAPI app created.")


## Step 4: Test the API from Inside the Notebook

We will use `TestClient` again so the notebook can exercise the endpoint directly.

In [ ]:
client = TestClient(app)


def call_query_api(payload: dict):
    response = client.post("/query", json=payload)
    print(f"Status: {response.status_code}")
    print(response.json())
    print("-" * 80)


print("Test client ready.")

### Example A: Call OpenAI Through the API

Run this cell only if `OPENAI_API_KEY` and `OPENAI_MODEL` are set.

In [ ]:
openai_payload = {
    "provider": "openai",
    "prompt": "Explain FastAPI in 2 short bullet points.",
    "system_prompt": "You are teaching a beginner Python team.",
}

if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
    print("Set OPENAI_API_KEY and OPENAI_MODEL in .env before running this example.")
else:
    call_query_api(openai_payload)

### Example B: Call Ollama Through the API

Run this cell only if Ollama is running locally and `OLLAMA_MODEL` is set.

In [ ]:
ollama_payload = {
    "provider": "ollama",
    "prompt": "Give me a one-sentence summary of what an API does.",
    "system_prompt": "Be concise.",
}

if not os.getenv("OLLAMA_MODEL"):
    print("Set OLLAMA_MODEL in .env before running this example.")
else:
    call_query_api(ollama_payload)

## What Changed from Day 1

Day 1 returned static data from memory.

Day 2 still uses a simple FastAPI endpoint, but now the endpoint calls a real model provider and returns generated text.

That becomes the foundation for later days where the API is containerized, token usage is measured, and retrieval is added.